In [1]:
!pip install nibabel tqdm --quiet

In [2]:
import os
import time
import random
import numpy as np
import nibabel as nib
from glob import glob
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from sklearn.model_selection import train_test_split

In [3]:
SEED = 42
IMG_SIZE = 128
EPOCHS = 10
BATCH_SIZE = 4
LR = 1e-4
NUM_WORKERS = 4
PATIENCE = 6

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

CUDA: True
GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [4]:
DATA_ROOT = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
TRAIN_PATH = os.path.join(DATA_ROOT, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

all_cases = sorted(glob(os.path.join(TRAIN_PATH, "BraTS20_Training_*")))
all_cases = [c for c in all_cases if os.path.exists(f"{c}/{os.path.basename(c)}_seg.nii")]

print("Total cases:", len(all_cases))

Total cases: 368


In [5]:
def load_and_preprocess(path):
    cid = os.path.basename(path)

    def load_mod(mod):
        vol = nib.load(f"{path}/{cid}_{mod}.nii").get_fdata().astype(np.float32)
        mask = vol != 0
        if mask.sum() > 0:
            vol[mask] = (vol[mask] - vol[mask].mean()) / (vol[mask].std() + 1e-8)
        t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(
            t,
            size=(IMG_SIZE, IMG_SIZE, IMG_SIZE),
            mode="trilinear",
            align_corners=False,
        )
        return t.squeeze().numpy().astype(np.float16)

    flair = load_mod("flair")
    t1 = load_mod("t1")
    t1ce = load_mod("t1ce")
    t2 = load_mod("t2")

    seg = nib.load(f"{path}/{cid}_seg.nii").get_fdata().astype(np.float32)
    t = torch.from_numpy(seg).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE, IMG_SIZE), mode="nearest")
    seg = t.squeeze().numpy()

    wt = (seg > 0).astype(np.float16)
    tc = ((seg == 1) | (seg == 4)).astype(np.float16)
    et = (seg == 4).astype(np.float16)

    image = np.stack([flair, t1, t1ce, t2], axis=0)
    label = np.stack([wt, tc, et], axis=0)
    return image, label


print("Preprocessing and caching all volumes to RAM...")
cache_start = time.time()

image_cache = {}
label_cache = {}

for path in tqdm(all_cases):
    img, lbl = load_and_preprocess(path)
    image_cache[path] = img
    label_cache[path] = lbl

cache_time = time.time() - cache_start
print(f"Cache built in {cache_time:.1f}s  ({cache_time/60:.1f}min)")

total_mb = sum(v.nbytes for v in image_cache.values()) + sum(v.nbytes for v in label_cache.values())
print(f"Total RAM used: {total_mb / 1e9:.2f} GB")

Preprocessing and caching all volumes to RAM...


  0%|          | 0/368 [00:00<?, ?it/s]

Cache built in 662.5s  (11.0min)
Total RAM used: 10.80 GB


In [6]:
class BraTSDataset(Dataset):
    def __init__(self, cases, augment=False):
        self.cases = cases
        self.augment = augment

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        path = self.cases[idx]
        image = torch.from_numpy(image_cache[path].astype(np.float32))
        label = torch.from_numpy(label_cache[path].astype(np.float32))

        if self.augment:
            for ax in [1, 2, 3]:
                if random.random() < 0.5:
                    image = torch.flip(image, [ax])
                    label = torch.flip(label, [ax])
            image = image * torch.empty(4, 1, 1, 1).uniform_(0.9, 1.1)
            image = image + torch.empty(4, 1, 1, 1).uniform_(-0.1, 0.1)

        return image, label

In [7]:
train_ids, temp_ids = train_test_split(all_cases, test_size=0.3, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=SEED)

print("Train:", len(train_ids), " Val:", len(val_ids), " Test:", len(test_ids))

train_loader = DataLoader(
    BraTSDataset(train_ids, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)
val_loader = DataLoader(
    BraTSDataset(val_ids, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)
test_loader = DataLoader(
    BraTSDataset(test_ids, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))

Train: 257  Val: 55  Test: 56
Train batches: 64
Val batches:   14


In [8]:
# ---------------------------------------------------------------------------
# UNETR — UNEt TRansformers (Hatamizadeh et al., 2021)
# Encoder  : ViT-Base  (patch_size=16, hidden_dim=768, 12 heads, 12 layers)
# Skips    : extracted at transformer layers 3, 6, 9, 12
# Decoder  : hierarchical CNN with ConvTranspose3d 2× upsampling
# ---------------------------------------------------------------------------


class PatchEmbedding3D(nn.Module):
    """Partition volume into non-overlapping patches and linearly project."""

    def __init__(self, in_channels=4, patch_size=16, hidden_dim=768, img_size=128):
        super().__init__()
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size  # 8
        self.proj = nn.Conv3d(in_channels, hidden_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, D, g, g, g)
        B, C, h, w, d = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, n_patches, D)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=12, mlp_dim=3072, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, hidden_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        y = self.norm1(x)
        y, _ = self.attn(y, y, y)
        x = x + y
        x = x + self.mlp(self.norm2(x))
        return x


class ConvBlock3D(nn.Module):
    """Two Conv3D-IN-ReLU layers."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNETR(nn.Module):
    """
    img_size=128, patch_size=16  →  grid_size=8  →  n_patches=512
    Extract transformer features at layers 3, 6, 9, 12.
    All zi have shape (B, 512, 768); reshaped to (B, 768, 8, 8, 8) for decoding.

    Decoder levels (resolution → resolution):
      L4:  8³  → 16³   uses z12 + z9
      L3:  16³ → 32³   uses z6
      L2:  32³ → 64³   uses z3
      L1:  64³ → 128³  uses raw input x
    """

    def __init__(
        self,
        in_channels=4,
        out_channels=3,
        img_size=128,
        patch_size=16,
        hidden_dim=768,
        num_heads=12,
        num_layers=12,
        mlp_dim=3072,
        dropout=0.1,
        extract_at=(3, 6, 9, 12),
    ):
        super().__init__()
        self.extract_at = set(extract_at)
        self.num_layers = num_layers
        self.grid_size = img_size // patch_size  # 8
        self.hidden_dim = hidden_dim

        # ---- Encoder -------------------------------------------------------
        self.patch_embed = PatchEmbedding3D(in_channels, patch_size, hidden_dim, img_size)
        n_patches = self.grid_size**3  # 512
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, hidden_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(dropout)

        self.transformer = nn.ModuleList(
            [TransformerBlock(hidden_dim, num_heads, mlp_dim, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(hidden_dim)

        # ---- Decoder -------------------------------------------------------
        D = hidden_dim

        # L4: 8³ → 16³  (z12 main path + z9 skip)
        self.l4_main_up = nn.ConvTranspose3d(D, 512, 2, 2)  # z12: 8→16
        self.l4_skip_proj = ConvBlock3D(D, 512)  # z9 at 8³
        self.l4_skip_up = nn.ConvTranspose3d(512, 512, 2, 2)  # z9 skip: 8→16
        self.l4_conv = ConvBlock3D(1024, 256)

        # L3: 16³ → 32³  (z6 skip via 2× deconv)
        self.l3_up = nn.ConvTranspose3d(256, 256, 2, 2)
        self.l3_skip_up = nn.Sequential(
            nn.ConvTranspose3d(D, 256, 2, 2),  # 8→16
            nn.ConvTranspose3d(256, 256, 2, 2),  # 16→32
        )
        self.l3_conv = ConvBlock3D(512, 128)

        # L2: 32³ → 64³  (z3 skip via 3× deconv)
        self.l2_up = nn.ConvTranspose3d(128, 128, 2, 2)
        self.l2_skip_up = nn.Sequential(
            nn.ConvTranspose3d(D, 256, 2, 2),  # 8→16
            nn.ConvTranspose3d(256, 128, 2, 2),  # 16→32
            nn.ConvTranspose3d(128, 128, 2, 2),  # 32→64
        )
        self.l2_conv = ConvBlock3D(256, 64)

        # L1: 64³ → 128³  (raw input skip)
        self.l1_up = nn.ConvTranspose3d(64, 64, 2, 2)
        self.l1_skip_proj = ConvBlock3D(in_channels, 64)  # raw x at 128³
        self.l1_conv = ConvBlock3D(128, 32)

        # Output
        self.out_conv = nn.Conv3d(32, out_channels, 1)

    def _reshape(self, z):
        """(B, n_patches, D) → (B, D, g, g, g)"""
        B, N, C = z.shape
        g = self.grid_size
        return z.transpose(1, 2).reshape(B, C, g, g, g)

    def forward(self, x):
        # ---- Encode --------------------------------------------------------
        tokens = self.pos_drop(self.patch_embed(x) + self.pos_embed)

        zs = {}
        for i, layer in enumerate(self.transformer):
            tokens = checkpoint(layer, tokens, use_reentrant=False)
            if (i + 1) in self.extract_at:
                zs[i + 1] = tokens

        z3 = self._reshape(zs[3])
        z6 = self._reshape(zs[6])
        z9 = self._reshape(zs[9])
        z12 = self._reshape(self.norm(zs[12]))

        # ---- Decode --------------------------------------------------------
        # L4: 8³ → 16³
        d4 = self.l4_conv(
            torch.cat(
                [
                    self.l4_main_up(z12),
                    self.l4_skip_up(self.l4_skip_proj(z9)),
                ],
                dim=1,
            )
        )  # (B, 256, 16,16,16)

        # L3: 16³ → 32³
        d3 = self.l3_conv(
            torch.cat(
                [
                    self.l3_up(d4),
                    self.l3_skip_up(z6),
                ],
                dim=1,
            )
        )  # (B, 128, 32,32,32)

        # L2: 32³ → 64³
        d2 = self.l2_conv(
            torch.cat(
                [
                    self.l2_up(d3),
                    self.l2_skip_up(z3),
                ],
                dim=1,
            )
        )  # (B,  64, 64,64,64)

        # L1: 64³ → 128³
        d1 = self.l1_conv(
            torch.cat(
                [
                    self.l1_up(d2),
                    self.l1_skip_proj(x),
                ],
                dim=1,
            )
        )  # (B,  32, 128,128,128)

        return self.out_conv(d1)

In [9]:
def region_loss(logits, target):
    pred = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(0, 2, 3, 4))
    union = pred.sum(dim=(0, 2, 3, 4)) + target.sum(dim=(0, 2, 3, 4))
    dice = 1 - (2 * inter + 1e-6) / (union + 1e-6)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    return dice.mean() + bce

In [10]:
model = UNETR()
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda")

print("Parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Parameters: 137,596,579


In [11]:
def dice_score(pred_bin, target_bin):
    inter = (pred_bin & target_bin).sum().float()
    return ((2 * inter + 1e-6) / (pred_bin.sum() + target_bin.sum() + 1e-6)).item()


def brats_metrics(pred_logits, target):
    pred = torch.sigmoid(pred_logits) > 0.5
    wt = dice_score(pred[:, 0], target[:, 0].bool())
    tc = dice_score(pred[:, 1], target[:, 1].bool())
    et = dice_score(pred[:, 2], target[:, 2].bool())
    return wt, tc, et

In [12]:
best_mean_dice = 0.0
patience_counter = 0
train_start = time.time()

for epoch in range(EPOCHS):
    t0 = time.time()
    current_lr = scheduler.get_last_lr()[0]
    elapsed = (t0 - train_start) / 60
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}   LR: {current_lr:.6f}   Elapsed: {elapsed:.1f}min")

    model.train()
    train_loss = 0.0
    for x, y in tqdm(train_loader, desc="Train", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            output = model(x)
            loss = region_loss(output, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        torch.cuda.empty_cache()

    scheduler.step()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    wt_scores, tc_scores, et_scores = [], [], []

    with torch.no_grad():
        for x, y in tqdm(val_loader, desc="Val", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast("cuda"):
                output = model(x)
                loss = region_loss(output, y)
            val_loss += loss.item()
            wt, tc, et = brats_metrics(output, y)
            wt_scores.append(wt)
            tc_scores.append(tc)
            et_scores.append(et)

    val_loss /= len(val_loader)
    wt_mean = np.mean(wt_scores)
    tc_mean = np.mean(tc_scores)
    et_mean = np.mean(et_scores)
    mean_dice = (wt_mean + tc_mean + et_mean) / 3
    epoch_time = time.time() - t0

    print(f"Train Loss: {train_loss:.4f}   Val Loss: {val_loss:.4f}")
    print(f"WT: {wt_mean:.4f}   TC: {tc_mean:.4f}   ET: {et_mean:.4f}   Mean: {mean_dice:.4f}")
    print(f"Epoch time: {epoch_time:.1f}s   Total elapsed: {(time.time()-train_start)/60:.1f}min")

    if mean_dice > best_mean_dice:
        best_mean_dice = mean_dice
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"Best model saved  (mean dice: {best_mean_dice:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nTraining complete. Total time: {(time.time()-train_start)/60:.1f}min")


Epoch 1/10   LR: 0.000100   Elapsed: 0.0min


Train:   0%|          | 0/64 [00:00<?, ?it/s]

Val:   0%|          | 0/14 [00:00<?, ?it/s]

Train Loss: 1.5717   Val Loss: 1.5129
WT: 0.5497   TC: 0.1266   ET: 0.0388   Mean: 0.2383
Epoch time: 714.4s   Total elapsed: 11.9min
Best model saved  (mean dice: 0.2383)

Epoch 2/10   LR: 0.000098   Elapsed: 11.9min


Train:   0%|          | 0/64 [00:00<?, ?it/s]

...

Val:   0%|          | 0/14 [00:00<?, ?it/s]

Train Loss: 1.3408   Val Loss: 1.3409
WT: 0.7217   TC: 0.6657   ET: 0.5230   Mean: 0.6368
Epoch time: 700.1s   Total elapsed: 105.5min
Best model saved  (mean dice: 0.6368)

Epoch 10/10   LR: 0.000003   Elapsed: 105.5min


Train:   0%|          | 0/64 [00:00<?, ?it/s]

Val:   0%|          | 0/14 [00:00<?, ?it/s]

Train Loss: 1.3384   Val Loss: 1.3397
WT: 0.7263   TC: 0.6691   ET: 0.5149   Mean: 0.6368
Epoch time: 701.2s   Total elapsed: 117.2min
No improvement (1/6)

Training complete. Total time: 117.2min


In [13]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_loss = 0.0
wt_scores, tc_scores, et_scores = [], [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda"):
            output = model(x)
            loss = region_loss(output, y)
        test_loss += loss.item()
        wt, tc, et = brats_metrics(output, y)
        wt_scores.append(wt)
        tc_scores.append(tc)
        et_scores.append(et)

test_loss /= len(test_loader)
wt_mean = np.mean(wt_scores)
tc_mean = np.mean(tc_scores)
et_mean = np.mean(et_scores)
mean_dice = (wt_mean + tc_mean + et_mean) / 3

print(f"\n{'='*60}")
print("FINAL TEST RESULTS")
print(f"Test Loss : {test_loss:.4f}")
print(f"WT Dice   : {wt_mean:.4f}")
print(f"TC Dice   : {tc_mean:.4f}")
print(f"ET Dice   : {et_mean:.4f}")
print(f"Mean Dice : {mean_dice:.4f}")

Test:   0%|          | 0/14 [00:00<?, ?it/s]


FINAL TEST RESULTS
Test Loss : 1.3411
WT Dice   : 0.7128
TC Dice   : 0.6261
ET Dice   : 0.5314
Mean Dice : 0.6234
